<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #19fff723; font-size:100%; border: 1px solid #ccc;"> 
<h1><strong> Problem </strong>    </h1>

<font color= #2d10ac>

The dataset used in this project is the Online Retail II Dataset, which contains transaction records from a UK-based non-store online retail business between December 2009 and December 2011. The company mainly specializes in unique all-occasion gift products, and a large portion of its customers are wholesalers.

This dataset provides detailed information about customer purchases, including invoice numbers, product descriptions, quantities, prices, transaction dates, and customer locations. Because of its rich transactional data, it is highly suitable for exploratory data analysis, customer analytics, sales forecasting, and business strategy development.

In this project, the dataset will be analyzed to better understand sales performance, customer purchasing behavior, and product trends. Several data mining and machine learning techniques will also be applied to generate business insights and support future sales strategies.

<h4><strong> Project Objectives </strong></h4>

1. **Sales Performance Analysis**  
   Evaluate overall sales performance by analyzing revenue trends, transaction patterns, top-selling products, and seasonal sales behavior.

2. **Customer RFM**  
   Evaluate customer purchasing behavior using Recency, Frequency, and Monetary (RFM) analysis to identify customer value and engagement levels.

3. **Customer Segmentation**  
   Group customers into meaningful segments based on purchasing behavior, spending habits, and transaction frequency to better understand different customer profiles.

4. **Product Sales Forecasting for 2012**  
   Build forecasting models to predict future product sales for the year 2012 using historical transaction data.

5. **Sales Improvement Strategy via Market Basket Analysis**  
   Identify relationships between frequently purchased products to support cross-selling opportunities, product bundling, and targeted marketing strategies.



</div>


<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #19fff723; font-size:100%; border: 1px solid #ccc;">

<h4><strong> About the dataset </strong></h4>

<font color= #2d10ac>

| Attribute | Type | Description |
|---|---|---|
| **InvoiceNo** | Nominal | Invoice number. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter `'C'`, it indicates a cancellation. |
| **StockCode** | Nominal | Product (item) code. A 5-digit integral number uniquely assigned to each distinct product. |
| **Description** | Nominal | Product (item) name. |
| **Quantity** | Numeric | The quantities of each product (item) per transaction. |
| **InvoiceDate** | Numeric | Invoice date and time. The day and time when a transaction was generated. |
| **UnitPrice** | Numeric | Unit price. Product price per unit in sterling (£). |
| **CustomerID** | Nominal | Customer number. A 5-digit integral number uniquely assigned to each customer. |
| **Country** | Nominal | Country name. The name of the country where a customer resides. |


</div>

<a id="contents-label"></a>    
<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #19fff723; font-size:100%; border: 1px solid #ccc;">
<h4 > <strong>Table of Contents:</h4>
<font color=#2d10ac>    

* [Data Transformation](#data-transformation)
* [Feature Engineering](#feature-engineering)
* [Customers Segmentation](#customers-segmentation)
* [Sales Forecasting](#sales-forecasting)
* [Customer Products Recommendation](#customer-products-recommendation)

<a id="data-transformation"></a>
# <p style="background-color: #81f1f3; font-family:calibri; color:dark; font-size:80%; font-family:Verdana; text-align:left;  padding: 20px;  border-radius:15px 15px;"> Data Transformation </p>

⬆️ [Tabel of Contents](#contents-label)

- Import library
- Fix data format
- Remove duplicates, nulls and cancelled invoices

In [1]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nltk
from nltk.corpus import stopwords
from collections import Counter
import re
import spacy
from tqdm import tqdm
import sys
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import matplotlib
import prophet
import nbformat

c:\Users\yam__\miniconda3\envs\sales_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
path=r'C:\online_retail 2'
# Load data (assuming the file is named 'online_retail_II.xlsx')
filename='online_retail_II.csv'

os.chdir(path)
df = pd.read_csv(filename)

In [4]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB
None


In [5]:
# 1. Date Conversion (Crucial for Seasonality Analysis)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 2. Numeric Conversions
# errors='coerce' turns non-numeric junk into NaN so it doesn't crash the script
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# 3. Handle CustomerID as Nullable Integers
# This keeps IDs like 17850 from looking like 17850.0
df['Customer ID'] = df['Customer ID'].astype('Int64')

# 4. String/Object Conversions for Nominal Data
nominal_cols = ['Invoice', 'StockCode', 'Description', 'Country']
for col in nominal_cols:
    df[col] = df[col].astype(str)

# Verify the changes
print(df.dtypes)

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                   str
dtype: object


In [6]:
# Check for duplicates and missing values
print(f"Duplicate rows: {df.duplicated().sum()}")

# Identify all duplicate rows (showing all copies)
duplicate_rows = df[df.duplicated(keep=False)]

# Display the first 3 duplicates to verify
print(f"Total duplicate rows found: {len(duplicate_rows)}")
print(duplicate_rows.head(3))

Duplicate rows: 34335
Total duplicate rows found: 67242
    Invoice StockCode                       Description  Quantity  \
362  489517     21913    VINTAGE SEASIDE JIGSAW PUZZLES         1   
363  489517     21912          VINTAGE SNAKES & LADDERS         1   
365  489517     21821  GLITTER STAR GARLAND WITH BELLS          1   

            InvoiceDate  Price  Customer ID         Country  
362 2009-12-01 11:34:00   3.75        16329  United Kingdom  
363 2009-12-01 11:34:00   3.75        16329  United Kingdom  
365 2009-12-01 11:34:00   3.75        16329  United Kingdom  


In [7]:
# Check for duplicates and missing values
print(f"Duplicate rows: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)

print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Missing values{df.isnull().sum()}")

Duplicate rows: 34335
Duplicate rows: 0
Missing valuesInvoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
Customer ID    235151
Country             0
dtype: int64


In [8]:
df.isnull().sum()[df.isnull().sum() > 0]

Description      4275
Customer ID    235151
dtype: int64

In [9]:
# 1. Drop rows with ANY null values and duplicate rows
df = df.dropna().drop_duplicates().reset_index()

# check for null and duplicate rows again
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Missing values{df.isnull().sum()}")


Duplicate rows: 0
Missing valuesindex          0
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


In [10]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 797885 entries, 0 to 797884
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   index        797885 non-null  int64         
 1   Invoice      797885 non-null  str           
 2   StockCode    797885 non-null  str           
 3   Description  797885 non-null  str           
 4   Quantity     797885 non-null  int64         
 5   InvoiceDate  797885 non-null  datetime64[us]
 6   Price        797885 non-null  float64       
 7   Customer ID  797885 non-null  Int64         
 8   Country      797885 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 94.8 MB
None


In [11]:
# Total Cost
df['TotalCost'] = df['Quantity'] * df['Price']

# Check and filter out cancellations for purchase behavior analysis
df_cancelled = df['Invoice'].astype(str).str.startswith('C')

print(f"Cancelled invoice rows: {df_cancelled.sum()}")

df_clean = df[~df_cancelled].copy().reset_index()


Cancelled invoice rows: 18390


In [12]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 779495 entries, 0 to 779494
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   level_0      779495 non-null  int64         
 1   index        779495 non-null  int64         
 2   Invoice      779495 non-null  str           
 3   StockCode    779495 non-null  str           
 4   Description  779495 non-null  str           
 5   Quantity     779495 non-null  int64         
 6   InvoiceDate  779495 non-null  datetime64[us]
 7   Price        779495 non-null  float64       
 8   Customer ID  779495 non-null  Int64         
 9   Country      779495 non-null  str           
 10  TotalCost    779495 non-null  float64       
dtypes: Int64(1), datetime64[us](1), float64(2), int64(3), str(4)
memory usage: 104.5 MB


<a id="feature-engineering"></a>
# <p style="background-color: #81f1f3; font-family:calibri; color:dark; font-size:80%; font-family:Verdana; text-align:left;  padding: 20px;  border-radius:15px 15px;"> Feature Engineering </p>

⬆️ [Tabel of Contents](#contents-label)

- Product category Classification using NLP approach
- Customer RFM Analysis

In [13]:
# Download necessary NLTK data
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def get_category(text):
    if not isinstance(text, str): return "OTHER"
    # Simple NLP: Remove punctuation, numbers, and stopwords
    words = re.sub(r'[^a-zA-Z\s]', '', text).lower().split()
    filtered = [w for w in words if w not in stop_words and len(w) > 2]
    # Return the first significant word as a proxy for 'Category'
    return filtered[0].upper() if filtered else "OTHER"

df_clean['Category_nltk'] = df_clean['Description'].apply(get_category)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yam__\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
df_clean['Category_nltk'].value_counts().head(10)

Category_nltk
SET        54285
RED        32564
JUMBO      24183
PACK       21929
LUNCH      19958
PINK       18883
VINTAGE    15398
WHITE      13923
BLUE       12703
WOODEN     11516
Name: count, dtype: int64

In [ ]:
nlp = spacy.load("en_core_web_sm")
#nlp = spacy.load("en_core_web_trf")

# Words that are styles or noise, NOT core categories
NOISE_WORDS = {
    'RETROSPOT', 'DESIGN', 'VINTAGE', 'OF', 'RED', 'BLUE', 'WHITE', 'PINK', 
    'SMALL', 'LARGE', 'OTHER', 'METAL', 'WOODEN', 'GLASS', 'PACK', 'BAG', 'HEART', 'NAN', 'GIRL', 'LONDON'    
    }

def refine_category(text):
    if not isinstance(text, str): return "MISC"
    
    doc = nlp(text.upper())
    
    # 1. Extract Noun Chunks
    chunks = list(doc.noun_chunks)
    if not chunks:
        # Fallback to last word if no chunk found
        words = [t.text for t in doc if t.pos_ in ['NOUN', 'PROPN']]
        candidate = words[-1] if words else "MISC"
    else:
        candidate = chunks[-1].text

    # 2. Clean the candidate
    # Remove noise words and punctuation
    clean_words = [w.text for w in nlp(candidate) if w.text not in NOISE_WORDS and not w.is_punct]
    
    # 3. Validation Logic
    # If our cleaning left us empty or with a generic word, 
    # search the whole description for any noun that isn't noise.
    if not clean_words or clean_words[-1] in NOISE_WORDS:
        all_nouns = [t.text for t in doc if t.pos_ in ['NOUN', 'PROPN'] and t.text not in NOISE_WORDS]
        if all_nouns:
            return all_nouns[-1] # Return the most specific noun found
        return "GIFTWARE" # Standard default for this dataset
        
    return " ".join(clean_words)

# --- Apply to unique values to save time ---
unique_desc = df_clean['Description'].dropna().unique()
tqdm.pandas(desc="Refining Categories")
category_map = {desc: refine_category(desc) for desc in tqdm(unique_desc)}

df_clean['Category'] = df_clean['Description'].map(category_map)

In [ ]:
# Check the top 20 categories to see if noise remains
print(df_clean['Category'].value_counts().head(20))

Category
JUMBO                            5945
T LIGHT HOLDER                   5778
WICKER                           5532
CHRISTMAS                        4492
SET                              4414
LUNCH                            4282
BOX                              4176
SUKI                             4043
CHARLOTTE                        3656
REGENCY CAKESTAND 3 TIER         3337
SAUCER                           3216
CUTLERY                          3068
SPOTS                            2742
CALM                             2697
ASSORTED COLOUR BIRD ORNAMENT    2692
BOWL                             2570
CARD                             2502
SPOT CERAMIC DRAWER KNOB         2358
POPCORN HOLDER                   2277
TISSUES                          2267
Name: count, dtype: int64



<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    
The generated category names are not fully effective for business analysis because many of them are too generic, ambiguous, or based on frequently occurring words rather than meaningful product groupings. Examples such as **"SET"**, **"BOX"**, **"SPOTS"**, **"CALM"**, and **"LUNCH"** do not clearly represent actual product categories and may lead to confusion during interpretation.

Some categories also capture partial product descriptions instead of semantic product classes. For instance, **"REGENCY CAKESTAND 3 TIER"** and **"ASSORTED COLOUR BIRD ORNAMENT"** are individual product names rather than broader categories. Similarly, terms like **"SUKI"** and **"CHARLOTTE"** appear to be branding or style keywords instead of functional product categories.

This issue commonly occurs when category names are generated using keyword frequency or text extraction methods without proper product taxonomy or natural language processing (NLP) refinement. As a result, the categories may not accurately reflect customer purchasing behavior or product relationships.

</div>

In [ ]:
from transformers import pipeline

# Initialize the zero-shot pipeline
# Note: This is a heavy model, use a GPU if available (device=0)
classifier = pipeline("zero-shot-classification", 
                      model="facebook/bart-large-mnli", 
                      device=0) # -1 is CPU, 0 is GPU

# A broad list of "Discovery Labels" to cover the Online Retail II catalog
candidate_labels = [
    "Kitchenware", "Home Decor", "Stationery", "Fashion Accessories", 
    "Children's Toys", "Garden & Outdoor", "Office Supplies", 
    "Gift Wrapping", "Lighting", "Bathroom Accessories"
]

def zero_shot_discovery(text):
    if not isinstance(text, str) or text.strip() == "":
        return "Uncategorized"
    
    # We use hypothesis_template to guide the model's logic
    result = classifier(text, 
                        candidate_labels, 
                        hypothesis_template="This item is a type of {}.")
    
    # Return the top-scoring label
    return result['labels'][0]

# --- Optimized Execution ---
# 1. Isolate unique descriptions (Crucial: 1M rows would take days, 4k uniques takes ~30 mins)
unique_desc = df_clean ['Description'].dropna().unique()

# 2. Process unique descriptions
processed_map = {}
for desc in tqdm(unique_desc, desc="Zero-Shot Classification"):
    processed_map[desc] = zero_shot_discovery(desc)

# 3. Map back to the main dataframe
df_clean['Category'] = df_clean['Description'].map(processed_map)

In [3]:
# Check the top 20 categories to see if noise remains
print(df_clean['Category'].value_counts().head(10))

Category
Home Decor              489675
Kitchenware             138403
Fashion Accessories      92829
Children's Toys          89408
Bathroom Accessories     64294
Lighting                 51623
Stationery               33605
Garden & Outdoor         19076
Office Supplies          18363
Gift Wrapping            16656
Name: count, dtype: int64


In [4]:
df_clean[['Description','Category']].head(10)

,Description,Category
0,15CM CHRISTMAS GLASS BALL 20 LIGHTS,Lighting
1,PINK CHERRY LIGHTS,Lighting
2,WHITE CHERRY LIGHTS,Lighting
3,"RECORD FRAME 7"" SINGLE SIZE",Home Decor
4,STRAWBERRY CERAMIC TRINKET BOX,Home Decor
5,PINK DOUGHNUT TRINKET POT,Kitchenware
6,SAVE THE PLANET MUG,Kitchenware
7,FANCY FONT HOME SWEET HOME DOORMAT,Home Decor
8,CAT BOWL,Bathroom Accessories
9,"DOG BOWL , CHASING BALL DESIGN",Children's Toys



<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    
The category classification results show a significant improvement after applying transformer-based language models compared to traditional NLP approaches such as SpaCy or NLTK.

Unlike keyword-based or rule-based methods, transformer models are able to understand the contextual meaning of product descriptions rather than relying only on individual word frequency. As a result, the generated categories are more structured, meaningful, and business-friendly.

The new categories such as **Home Decor**, **Kitchenware**, **Fashion Accessories**, **Children's Toys**, and **Bathroom Accessories** represent clear and interpretable product groups that are easier to analyze for sales trends, customer preferences, and business decision-making.

Compared to the earlier results generated using SpaCy or NLTK:
- Generic terms such as *"SET"*, *"BOX"*, and *"SPOTS"* are no longer dominant.
- Product-specific phrases are successfully grouped into broader semantic categories.
- The classification output is more consistent and less noisy.
- Categories are more aligned with real-world retail taxonomy.

Transformer models perform better because they capture semantic relationships and contextual information within product descriptions. For example, products containing different wording but similar meanings can still be classified into the same category. This is something traditional NLP methods often struggle with, especially when dealing with short retail product descriptions.

Overall, the transformer-based approach produces cleaner, more scalable, and commercially meaningful categories, making the dataset more suitable for downstream tasks such as:
- customer segmentation,
- recommendation systems,
- market basket analysis,
- sales forecasting, and
- inventory planning.

</div>

In [5]:
# Reference date for Recency (Day after the last invoice)
ref_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# Group by Customer
customer_data = df_clean.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (ref_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalCost': 'sum'
}).rename(columns={'InvoiceDate': 'Recency', 'Invoice': 'Frequency', 'TotalCost': 'Monetary'})

In [6]:
# Map RFM features back to original dataframe
df_clean = df_clean.merge(
    customer_data,
    on='Customer ID',
    how='left'
)

# Preview
df_clean[['Customer ID', 'Recency', 'Frequency', 'Monetary']].head()

,Customer ID,Recency,Frequency,Monetary
0,13085,158.0,8.0,2433.28
1,13085,158.0,8.0,2433.28
2,13085,158.0,8.0,2433.28
3,13085,158.0,8.0,2433.28
4,13085,158.0,8.0,2433.28


In [7]:
# Create a figure with 3 side-by-side subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Recency (Days)", "Frequency (Orders)", "Monetary (£ Value)"),
    horizontal_spacing=0.1
)

# 1. Recency Box Plot
fig.add_trace(go.Box(y=customer_data['Recency'], name='Recency', boxpoints='outliers'), row=1, col=1)

# 2. Frequency Box Plot 
fig.add_trace(go.Box(y=customer_data['Frequency'], name='Frequency', boxpoints='outliers'), row=1, col=2)

# 3. Monetary Box Plot
fig.add_trace(go.Box(y=customer_data['Monetary'], name='Monetary', boxpoints='outliers'), row=1, col=3)

# Update layout for a professional look
fig.update_layout(
    title_text="RFM Metric Distribution & Outlier Detection",
    template="plotly_white",
    showlegend=False,
    height=500
)

# Optional: Use log scale for Monetary if you have extreme 'Whale' customers
# fig.update_yaxes(type="log", row=1, col=3)

fig.show()

In [8]:
# Create a figure with 3 side-by-side subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Recency (Days)", "Frequency (Orders)", "Monetary (£ Value)"),
    horizontal_spacing=0.1
)

# 1. Recency Box Plot
fig.add_trace(go.Box(y=customer_data['Recency'], name='Recency', boxpoints='outliers'), row=1, col=1)

# 2. Frequency Box Plot 
fig.add_trace(go.Box(y=customer_data['Frequency'], name='Frequency', boxpoints=False), row=1, col=2)

# 3. Monetary Box Plot
fig.add_trace(go.Box(y=customer_data['Monetary'], name='Monetary', boxpoints=False), row=1, col=3)

# Update layout for a professional look
fig.update_layout(
    title_text="RFM Metric Distribution & Outlier Detection",
    template="plotly_white",
    showlegend=False,
    height=500
)

# Optional: Use log scale for Monetary if you have extreme 'Whale' customers
#fig.update_yaxes(type="log", row=1, col=2)
#fig.update_yaxes(type="log", row=1, col=3)

fig.show()

In [9]:
# Compute boxplot statistics for RFM features

rfm_cols = ['Recency', 'Frequency', 'Monetary']

for col in rfm_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q2 = df_clean[col].median()
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)][col].count()

    print(f"\n===== {col} =====")
    print(f"Minimum        : {df_clean[col].min():.2f}")
    print(f"Q1 (25%)       : {Q1:.2f}")
    print(f"Median (50%)   : {Q2:.2f}")
    print(f"Q3 (75%)       : {Q3:.2f}")
    print(f"Maximum        : {df_clean[col].max():.2f}")
    print(f"IQR             : {IQR:.2f}")
    print(f"Lower Bound     : {lower_bound:.2f}")
    print(f"Upper Bound     : {upper_bound:.2f}")
    print(f"Number Outliers : {outliers}")


===== Recency =====
Minimum        : 1.00
Q1 (25%)       : 6.00
Median (50%)   : 22.00
Q3 (75%)       : 74.00
Maximum        : 739.00
IQR             : 68.00
Lower Bound     : -96.00
Upper Bound     : 176.00
Number Outliers : 118468

===== Frequency =====
Minimum        : 1.00
Q1 (25%)       : 5.00
Median (50%)   : 12.00
Q3 (75%)       : 26.00
Maximum        : 398.00
IQR             : 21.00
Lower Bound     : -26.50
Upper Bound     : 57.50
Number Outliers : 83504

===== Monetary =====
Minimum        : 0.00
Q1 (25%)       : 1728.99
Median (50%)   : 4467.66
Q3 (75%)       : 11383.03
Maximum        : 580987.04
IQR             : 9654.04
Lower Bound     : -12752.07
Upper Bound     : 25864.09
Number Outliers : 101997


In [10]:
# Create subplots: 1 row, 3 columns
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=("Recency Distribution", "Frequency Distribution", "Monetary Distribution"),
    horizontal_spacing=0.08
)

# 1. Recency Histogram (Days)
fig.add_trace(
    go.Histogram(x=customer_data['Recency'], name='Recency', marker_color='#636EFA', nbinsx=30),
    row=1, col=1
)

# 2. Frequency Histogram (Count of Invoices)
fig.add_trace(
    go.Histogram(x=customer_data['Frequency'], name='Frequency', marker_color='#EF553B', nbinsx=30),
    row=1, col=2
)

# 3. Monetary Histogram (Total Spend)
fig.add_trace(
    go.Histogram(x=customer_data['Monetary'], name='Monetary', marker_color='#00CC96', nbinsx=30),
    row=1, col=3
)

# Update layout for aesthetics
fig.update_layout(
    title_text="Customer Behavior Density Analysis (RFM Histograms)",
    template="plotly_white",
    showlegend=False,
    height=500
)

# Add X-axis labels
fig.update_xaxes(title_text="Days Since Last Purchase", row=1, col=1)
fig.update_xaxes(title_text="Number of Invoices", row=1, col=2)
fig.update_xaxes(title_text="Total Spending (£)", row=1, col=3)

fig.show()

<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>

The boxplot statistics for the RFM (Recency, Frequency, Monetary) features reveal that the dataset contains a substantial number of outliers and is highly skewed, which is common in real-world retail transaction data.

#### Recency
The **Recency** feature shows that:
- 25% of customers made purchases within 5 days,
- the median customer purchased within 21 days, and
- 75% of customers purchased within 73 days.

However, the maximum recency value reaches 739 days, indicating that some customers have been inactive for a very long period. The large number of outliers suggests the presence of dormant or one-time customers who stopped purchasing long ago.

The negative lower bound is not practically meaningful because recency values cannot be negative. This indicates that the distribution is strongly right-skewed.

#### Frequency
The **Frequency** statistics indicate that most customers purchase relatively infrequently:
- median purchase frequency is 14 transactions,
- while some customers purchased up to 510 times.

The large number of high-frequency outliers likely represents loyal customers, wholesalers, or bulk buyers. This suggests that customer purchasing behavior is highly uneven, with a small group contributing disproportionately large transaction volumes.

Again, the negative lower bound is not meaningful because transaction frequency cannot be below zero.

#### Monetary
The **Monetary** feature exhibits the strongest variability among all RFM variables. The extremely large range and high number of outliers indicate that customer spending behavior differs significantly across the customer base.

The presence of negative monetary values suggests refunds, returns, cancellations, or credit adjustments in the dataset. High positive outliers likely correspond to premium customers or wholesale buyers with very large purchase amounts.

The large interquartile range (IQR) further confirms that customer spending is highly dispersed and right-skewed.

#### Overall Interpretation
Overall, the RFM variables demonstrate:
- highly skewed distributions,
- substantial customer heterogeneity,
- strong presence of extreme-value customers, and
- realistic retail purchasing behavior.

At the business level, the outliers are also valuable because they may represent:
- highly loyal customers,
- high-value wholesale buyers, or
- inactive customers at risk of churn.


</div>


# <a id="customers-segmentation"></a> <p style="background-color: #81f1f3; font-family:calibri; color:dark; font-size:80%; font-family:Verdana; text-align:left;  padding: 20px;  border-radius:15px 15px;"> Customers Segmentation </p>

⬆️ [Tabel of Contents](#contents-label)

- Normal vs Outlier Customers Purchasing Behavior

In [11]:
# 1. Define Outlier Thresholds using IQR for Frequency and Monetary
def get_outlier_thresholds(dataframe, variable):
    Q1 = dataframe[variable].quantile(0.25)
    Q3 = dataframe[variable].quantile(0.75)
    IQR = Q3 - Q1
    upper_limit = Q3 + 1.5 * IQR
    return upper_limit

freq_limit = get_outlier_thresholds(customer_data, 'Frequency')
monetary_limit = get_outlier_thresholds(customer_data, 'Monetary')

# 2. Segment Customers
# We define an "Outlier Customer" as anyone who exceeds the limit in EITHER Frequency OR Monetary
customer_data['Segment'] = 'Normal'
customer_data.loc[
    (customer_data['Frequency'] > freq_limit) | (customer_data['Monetary'] > monetary_limit), 
    'Segment'
] = 'High-Value Outlier'

# 3. Compute Stats: Count and Total Earnings
segment_analysis = customer_data.groupby('Segment').agg(
    Customer_Count=('Monetary', 'count'),
    Total_Earnings=('Monetary', 'sum'),
    Avg_Spend_Per_Customer=('Monetary', 'mean')
).reset_index()

# Calculate percentages for context
total_revenue = segment_analysis['Total_Earnings'].sum()
total_customers = segment_analysis['Customer_Count'].sum()

segment_analysis['Revenue_Contribution_%'] = (segment_analysis['Total_Earnings'] / total_revenue) * 100
segment_analysis['Customer_Base_%'] = (segment_analysis['Customer_Count'] / total_customers) * 100

print(segment_analysis)

              Segment  Customer_Count  Total_Earnings  Avg_Spend_Per_Customer  \
0  High-Value Outlier             696    1.158690e+07            16647.841261   
1              Normal            5185    5.787907e+06             1116.279026   

   Revenue_Contribution_%  Customer_Base_%  
0               66.687931        11.834722  
1               33.312069        88.165278  


In [12]:
import plotly.graph_objects as go

# We structure the data to group metrics on the X-axis and Segments in the Legend
fig = go.Figure()

# --- Customer Count (Left Axis) ---
fig.add_trace(go.Bar(
    name='Normal Customers',
    x=['Customer Count'],
    y=segment_analysis.loc[segment_analysis['Segment'] == 'Normal', 'Customer_Count'],
    marker_color='#636EFA',
    yaxis='y1',
    offsetgroup=0
))

fig.add_trace(go.Bar(
    name='High-Value Outliers',
    x=['Customer Count'],
    y=segment_analysis.loc[segment_analysis['Segment'] == 'High-Value Outlier', 'Customer_Count'],
    marker_color='#EF553B',
    yaxis='y1',
    offsetgroup=1
))

# --- Total Earnings (Right Axis) ---
fig.add_trace(go.Bar(
    name='Normal Earnings',
    x=['Total Earnings'],
    y=segment_analysis.loc[segment_analysis['Segment'] == 'Normal', 'Total_Earnings'],
    marker_color='#636EFA',
    opacity=0.6, # Differentiate earnings from count visually
    yaxis='y2',
    offsetgroup=0,
    showlegend=False # Keep legend clean
))

fig.add_trace(go.Bar(
    name='Outlier Earnings',
    x=['Total Earnings'],
    y=segment_analysis.loc[segment_analysis['Segment'] == 'High-Value Outlier', 'Total_Earnings'],
    marker_color='#EF553B',
    opacity=0.6,
    yaxis='y2',
    offsetgroup=1,
    showlegend=False
))

# --- Layout Configuration ---
fig.update_layout(
    title='Impact Analysis: Customer Count vs. Total Revenue Contribution',
    yaxis=dict(
        title='Number of Customers (Left)',
        side='left'
    ),
    yaxis2=dict(
        title='Total Earnings (£) (Right)',
        side='right',
        overlaying='y',
        showgrid=False
    ),
    barmode='group',
    template='plotly_white',
    legend_title_text='Customer Segment',
    height=600
)

fig.show()

<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>

The Bar chart shows an example of "Pareto Principle", where a small group of customers drives the most revenue.

Thge "High-Value Outliers" make up only 5–10% of your customer count but contribute 40–60% of your total revenue.

</div>

In [13]:
# 1. Join the Segment back to the original clean transaction dataframe
df_segment = df_clean.merge(customer_data[['Segment']], on='Customer ID', how='left')

# 2. Get top 5 categories for each segment
top_cats_segment = df_segment.groupby(['Segment', 'Category'])['TotalCost'].sum().reset_index()
top_cats_segment = top_cats_segment.sort_values(['Segment', 'TotalCost'], ascending=[True, False])

# Display the findings
for segment in ['Normal', 'High-Value Outlier']:
    print(f"\n--- Top 5 Categories for {segment} Customers ---")
    print(top_cats_segment[top_cats_segment['Segment'] == segment].head(5))


--- Top 5 Categories for Normal Customers ---
   Segment             Category    TotalCost
15  Normal           Home Decor  2905810.839
16  Normal          Kitchenware   817563.420
12  Normal  Fashion Accessories   443773.510
11  Normal      Children's Toys   438872.750
17  Normal             Lighting   378685.580

--- Top 5 Categories for High-Value Outlier Customers ---
              Segment             Category    TotalCost
5  High-Value Outlier           Home Decor  5649793.108
6  High-Value Outlier          Kitchenware  1773257.080
1  High-Value Outlier      Children's Toys   908802.830
2  High-Value Outlier  Fashion Accessories   849729.460
7  High-Value Outlier             Lighting   717577.100


In [14]:
# Aggregate sales by Segment and Category
cat_comparison = df_segment.groupby(['Segment', 'Category'])['TotalCost'].sum().reset_index()

# Get Top 5 for each segment
top_normal = cat_comparison[cat_comparison['Segment'] == 'Normal'].nlargest(10, 'TotalCost')
top_outliers = cat_comparison[cat_comparison['Segment'] == 'High-Value Outlier'].nlargest(10, 'TotalCost')

# Combine for plotting
plot_df = pd.concat([top_normal, top_outliers])

# Visualize with a Horizontal Bar Chart
fig = px.bar(plot_df, 
             x='TotalCost', 
             y='Category', 
             color='Segment',
             barmode='group',
             orientation='h',
             title='Top 10 Product Categories: Normal vs. High-Value Outliers',
             labels={'TotalCost': 'Total Spending (£)', 'Category': 'Product Category'},
             template='plotly_white')

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>

The results show a clear distinction between the **Normal** customer segment and the **High-Value Outlier** segment in terms of total sales contribution across product categories.

Across all categories, the **High-Value Outlier** customers consistently generate significantly higher revenue compared to Normal customers. This indicates that a relatively small group of customers contributes a disproportionately large share of total sales, which is a common pattern in retail businesses.

- **Home Decor** is the top-performing category for both customer segments:
  - Normal customers contributed approximately **2.94 million** in sales.
  - High-Value Outlier customers contributed approximately **5.62 million**, nearly double the Normal segment.

- **Kitchenware** is the second-largest contributor in both segments, showing strong and consistent demand across customer groups.

- Categories such as **Fashion Accessories**, **Children’s Toys**, and **Lighting** also generate substantial revenue, indicating diversified purchasing interests among customers.

- Lower-performing categories include:
  - **Gift Wrapping**
  - **Garden & Outdoor**
  - **Office Supplies**

  Although these categories contribute less revenue overall, they may still provide opportunities for targeted promotions or seasonal campaigns.

The High-Value Outlier segment appears to represent:
- loyal premium customers,
- wholesale buyers, or
- bulk purchasers with exceptionally high spending behavior.

These customers are strategically important because they contribute a major portion of total revenue despite likely representing a smaller percentage of the customer base.
o
Meanwhile, the Normal segment provides a more stable and broader revenue foundation, representing regular retail purchasing behavir.

</div>

In [15]:
## Deeper analysis into top performing category

# Filter and Calculate Total Earnings at the row level first
home_decor_df = df_segment[df_segment['Category'] == 'Home Decor'].copy()
home_decor_df['Earnings'] = home_decor_df['Quantity'] * home_decor_df['UnitPrice']

# Group by product and segment
product_stats = home_decor_df.groupby(['Description', 'Segment']).agg(
    Total_Earnings=('Earnings', 'sum'),
    Total_Units=('Quantity', 'sum'),
    Avg_Price=('UnitPrice', 'mean')
).reset_index()

# Identify Top 10 products based on TOTAL EARNINGS (Normal + Outlier combined)
top_10_earnings = product_stats.groupby('Description')['Total_Earnings'].sum().nlargest(10).index
plot_df = product_stats[product_stats['Description'].isin(top_10_earnings)].copy()

# Sort by Total Earnings for the horizontal bars (Descending from top)
sort_order = product_stats.groupby('Description')['Total_Earnings'].sum().sort_values(ascending=True).index
plot_df['Description'] = pd.Categorical(plot_df['Description'], categories=sort_order, ordered=True)
plot_df = plot_df.sort_values('Description')

# Create 3-Column Subplots
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=("Total Earnings (£)", "Total Units", "Avg Unit Price (£)"),
    shared_yaxes=True, 
    horizontal_spacing=0.04
)

colors = {'Normal': '#636EFA', 'High-Value Outlier': '#EF553B'}

for segment in ['Normal', 'High-Value Outlier']:
    seg_data = plot_df[plot_df['Segment'] == segment]
    
    # Left Plot: Total Earnings
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Total_Earnings'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=True
    ), row=1, col=1)
    
    # Middle Plot: Total Units
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Total_Units'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=False
    ), row=1, col=2)
    
    # Right Plot: Average Price
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Avg_Price'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=False
    ), row=1, col=3)

fig.update_layout(
    title_text="Top 10 Home Decor Products by Revenue: Earnings vs. Volume vs. Price",
    barmode='group', 
    template='plotly_white', 
    height=800,
    legend=dict(title="Segment", x=1.02, y=1)
)

fig.show()

In [16]:
# Assuming 'plot_df' was created in the previous step
# table_df = plot_df[['Description', 'Segment', 'Total_Earnings', 'Total_Units', 'Avg_Price']].sort_values(['Total_Earnings'], ascending=False)
# print(table_df.to_markdown(index=False))

<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>

1. White Hanging Heart is the Hero (High Volume + High Revenue). This is the core business driver.

2. Medium Ceramic Jar	is the Bulk	(Extreme Volume + Low Price). This is a "filler" item that keeps outliers coming back.

3. "Manual" usually represents custom service fees, shipping corrections, or manual credit/debit adjustments. This is administrative/service income and add "Noise" to the data. 

</div>

In [17]:
# Identify the Top 10 products based on TOTAL volume (both segments combined)

top_10_volume = product_stats.groupby('Description')['Total_Units'].sum().nlargest(10).index
plot_df = product_stats[product_stats['Description'].isin(top_10_volume)].copy()

# Sort the dataframe so the horizontal bars appear in descending order of total volume
# We create a sorting key based on the sum of units per product
sort_order = product_stats.groupby('Description')['Total_Units'].sum().sort_values(ascending=True).index
plot_df['Description'] = pd.Categorical(plot_df['Description'], categories=sort_order, ordered=True)
plot_df = plot_df.sort_values('Description')

# Create Subplots
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=("Total Earnings (£)", "Total Units", "Avg Unit Price (£)"),
    shared_yaxes=True, 
    horizontal_spacing=0.07
)

# Colors for segments
colors = {'Normal': '#636EFA', 'High-Value Outlier': '#EF553B'}

for segment in ['Normal', 'High-Value Outlier']:
    seg_data = plot_df[plot_df['Segment'] == segment]
    
    # Left Plot: Total Earnings
    fig.add_trace(go.Bar(
        y=seg_data['Description'], 
        x=seg_data['Total_Earnings'], 
        name=segment, 
        orientation='h', 
        marker_color=colors[segment],
        legendgroup=segment,
        showlegend=True 
    ), row=1, col=1)
    
    # Middle Plot: Total Units
    fig.add_trace(go.Bar(
        y=seg_data['Description'], 
        x=seg_data['Total_Units'], 
        name=segment, 
        orientation='h',
        marker_color=colors[segment],
        legendgroup=segment, showlegend=False
    ), row=1, col=2)
    
    # Right Plot: Average Price
    fig.add_trace(go.Bar(
        y=seg_data['Description'], 
        x=seg_data['Avg_Price'], 
        name=segment, 
        orientation='h', 
        marker_color=colors[segment],
        legendgroup=segment, showlegend=False
    ), row=1, col=3)


fig.update_layout(
    title_text="Top 10 Home Decor Products: Volume (Sorted) vs. Price",
    barmode='group', 
    template='plotly_white', 
    height=700,
    legend=dict(title="Segment", x=1.05, y=1)
)

<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    

1. The price difference is negligible (less than 5% in most cases), and these items are "commodities" (low-cost basics like gliders and cake cases). When prices are that stable across segments, this means the business has a flat pricing strategy regardless of volume.

2. Interestingly, __WW2 Gliders Design__ has the highest orders but is not one of the top earning due to its low price. 
</div>


In [18]:
## Deeper Analysis into second most categroy Kitchenware

kitchen_df = df_segment[df_segment['Category'] == 'Kitchenware'].copy()

# 2. Group by product and segment
# Includes unique Invoice count to show how many times the product was 'ordered'
product_stats = kitchen_df.groupby(['Description', 'Segment']).agg(
    Total_Earnings=('TotalCost', 'sum'),
    Total_Units=('Quantity', 'sum'),
    Avg_Price=('UnitPrice', 'mean'),
    Order_Count=('Invoice', 'nunique')
).reset_index()

# Identify Top 10 products based on TOTAL EARNINGS
top_10_kitchen = product_stats.groupby('Description')['Total_Earnings'].sum().nlargest(10).index
plot_df = product_stats[product_stats['Description'].isin(top_10_kitchen)].copy()

# Sort by Total Earnings for the plot
sort_order = product_stats.groupby('Description')['Total_Earnings'].sum().sort_values(ascending=True).index
plot_df['Description'] = pd.Categorical(plot_df['Description'], categories=sort_order, ordered=True)
plot_df = plot_df.sort_values('Description')

# Create 4-Column Subplots
fig = make_subplots(
    rows=1, cols=4, 
    subplot_titles=("Total Earnings (£)", "Total Units", "Avg Unit Price (£)", "Unique Orders"),
    shared_yaxes=True, 
    horizontal_spacing=0.03
)

colors = {'Normal': '#636EFA', 'High-Value Outlier': '#EF553B'}

for segment in ['Normal', 'High-Value Outlier']:
    seg_data = plot_df[plot_df['Segment'] == segment]
    
    # Column 1: Earnings
    fig.add_trace(go.Bar(y=seg_data['Description'], x=seg_data['Total_Earnings'], 
                         name=segment, orientation='h', marker_color=colors[segment],
                         legendgroup=segment, showlegend=True), row=1, col=1)
    
    # Column 2: Units
    fig.add_trace(go.Bar(y=seg_data['Description'], x=seg_data['Total_Units'], 
                         orientation='h', marker_color=colors[segment],
                         legendgroup=segment, showlegend=False), row=1, col=2)
    
    # Column 3: Price
    fig.add_trace(go.Bar(y=seg_data['Description'], x=seg_data['Avg_Price'], 
                         orientation='h', marker_color=colors[segment],
                         legendgroup=segment, showlegend=False), row=1, col=3)
    
    # Column 4: Order Count
    fig.add_trace(go.Bar(y=seg_data['Description'], x=seg_data['Order_Count'], 
                         orientation='h', marker_color=colors[segment],
                         legendgroup=segment, showlegend=False), row=1, col=4)

fig.update_layout(
    title_text="Top 10 Kitchenware Products: Revenue, Volume, Pricing, and Frequency",
    barmode='group', 
    template='plotly_white', 
    height=800,
    legend=dict(title="Segment", x=1.02, y=1)
)

fig.show()

<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    
Kitchenware features a true "Power Product"— the __Regency Cakestand 3 Tier__, an expensive outlier that completely changes the logistical scale. Outliers spent over £214k on this single item. Notably, their average price (£12.30) is slightly lower than Normal customers (£12.63), showing a clear volume incentive on a high-ticket item.

__PICNIC BASKET WICKER 60 PIECES__ : __The Ultra-High Ticket__. Only 2 unique orders from Outliers, but totaling £39.6k. This is a massive bulk-buy of a luxury kit (£649.50/unit) likely for a corporate event or high-end retail stocking.

 __Frequency vs. Volume__: Outliers: 1,804 orders for 18,894 units (~10 units per order). Normal: 1,514 orders for 5,245 units (~3 units per order). This shows High-Value Outliers for this specific product aren't just one-time bulk buyers; they are consistent repeat purchasers who order significantly larger quantities than the average user every time they visit.

</div>



In [19]:
# Data Cleaning
# Strip out administrative/service codes
df_segment_clean = df_segment[df_segment['Description'] != 'Manual'].copy()

# Recalculate Earnings for physical products only
df_segment_clean['Earnings'] = df_segment_clean['Quantity'] * df_segment_clean['UnitPrice']

In [20]:
# Aggregate by Segment and Category
cat_stats = df_segment_clean.groupby(['Category', 'Segment']).agg(
    Avg_UnitPrice=('UnitPrice', 'mean'),
    Avg_Qty=('Quantity', 'mean'),
    Total_Rev=('TotalCost', 'sum')
).reset_index()

# Filter for categories that both segments actually buy to ensure a fair comparison
common_cats = cat_stats.groupby('Category').filter(lambda x: x['Segment'].nunique() > 1)

# Get the top 10 categories by total revenue to keep the visualization clean
top_10_rev_cats = common_cats.groupby('Category')['Total_Rev'].sum().nlargest(10).index
plot_df = common_cats[common_cats['Category'].isin(top_10_rev_cats)]

# Visualize Avg Unit Price per Category by Segment
fig = px.bar(plot_df, 
             x='Category', 
             y='Avg_UnitPrice', 
             color='Segment',
             barmode='group',
             title='Average Unit Price per Category: Normal vs. Outlier',
             labels={'Avg_UnitPrice': 'Avg Price (£)'},
             template='plotly_white')

fig.show()

<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    
    
 While most retail categories show a standard bulk-discount pattern (Price Ratio < 1.0), Outliers in Stationery are paying a 54% premium compared to normal customers.
 
 The data suggests a striking "Premium Gap" in the Stationery and Home Decor categories.

</div>


In [21]:
# Pivot to calculate the ratio
#pivot_cats = plot_df.pivot(index='Category', columns='Segment', values='Avg_UnitPrice')
#pivot_cats['Price_Ratio'] = pivot_cats['High-Value Outlier'] / pivot_cats['Normal']

# Sort by Price_Ratio to see where the biggest difference lies
#print("--- Categories where Outliers pay a Premium ---")
#print(pivot_cats.sort_values('Price_Ratio', ascending=False))

In [22]:
## deeper analysis into staionery category

# Filter for Stationery using the cleaned product-only dataframe
stationery_df = df_segment_clean[df_segment_clean['Category'] == 'Stationery'].copy()

# 2. Group by product and segment
# Note: 'Earnings' was calculated in the previous step (Quantity * UnitPrice)
product_stats = stationery_df.groupby(['Description', 'Segment']).agg(
    Total_Earnings=('Earnings', 'sum'),
    Total_Units=('Quantity', 'sum'),
    Avg_Price=('UnitPrice', 'mean')
).reset_index()

# 3. Identify Top 10 products based on TOTAL EARNINGS (Normal + Outlier combined)
top_10_earnings_names = product_stats.groupby('Description')['Total_Earnings'].sum().nlargest(10).index
plot_df = product_stats[product_stats['Description'].isin(top_10_earnings_names)].copy()

# 4. Sort by Total Earnings for the horizontal bars (Ascending for the plot to show highest at top)
sort_order = product_stats.groupby('Description')['Total_Earnings'].sum().sort_values(ascending=True).index
plot_df['Description'] = pd.Categorical(plot_df['Description'], categories=sort_order, ordered=True)
plot_df = plot_df.sort_values('Description')

# 5. Create 3-Column Subplots
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=("Total Earnings (£)", "Total Units", "Avg Unit Price (£)"),
    shared_yaxes=True, 
    horizontal_spacing=0.04
)

colors = {'Normal': '#636EFA', 'High-Value Outlier': '#EF553B'}

for segment in ['Normal', 'High-Value Outlier']:
    seg_data = plot_df[plot_df['Segment'] == segment]
    
    # Left Plot: Total Earnings (Sorted)
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Total_Earnings'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=True
    ), row=1, col=1)
    
    # Middle Plot: Total Units
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Total_Units'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=False
    ), row=1, col=2)
    
    # Right Plot: Average Price
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Avg_Price'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=False
    ), row=1, col=3)

fig.update_layout(
    title_text="Top 10 Stationery Products by Revenue: Earnings vs. Volume vs. Price",
    barmode='group', 
    template='plotly_white', 
    height=800,
    legend=dict(title="Segment", x=1.02, y=1)
)

fig.show()

In [23]:
# check number of order for paper craft, little birdie
stationery_df[stationery_df['Description']=='PAPER CRAFT , LITTLE BIRDIE']['Invoice'].unique()

array(['581483'], dtype=object)


<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>

We have identified another set of __"Service/Non-Product"__ codes that are dominating the revenue: __POSTAGE__ and __DOTCOM POSTAGE__.  Similar to "Manual," these are fulfillment fees. They appear at the top because high-value outliers likely pay for heavy freight or international shipping.

Additionally, we found a massive outlier in the physical goods: __PAPER CRAFT, LITTLE BIRDIE__. This is a massive single transaction wholesale clearance that skews everything.

</div>


In [24]:
# Excluding the service product and paper craft birdie
exclude_list = ['Manual', 'POSTAGE', 'DOTCOM POSTAGE', 'PAPER CRAFT , LITTLE BIRDIE']

df_segment_clean = df_segment_clean[~df_segment_clean['Description'].isin(exclude_list)].reset_index(drop=True)
stationery_df_exclude = stationery_df[~stationery_df['Description'].isin(exclude_list)].reset_index(drop=True)

# Group by product and segment
# Note: 'Earnings' was calculated in the previous step (Quantity * UnitPrice)
product_stats = stationery_df_exclude.groupby(['Description', 'Segment']).agg(
    Total_Earnings=('Earnings', 'sum'),
    Total_Units=('Quantity', 'sum'),
    Avg_Price=('UnitPrice', 'mean')
).reset_index()

# Identify Top 10 products based on TOTAL EARNINGS (Normal + Outlier combined)
top_10_earnings_names = product_stats.groupby('Description')['Total_Earnings'].sum().nlargest(10).index
plot_df = product_stats[product_stats['Description'].isin(top_10_earnings_names)].copy()

# Sort by Total Earnings for the horizontal bars (Ascending for the plot to show highest at top)
sort_order = product_stats.groupby('Description')['Total_Earnings'].sum().sort_values(ascending=True).index
plot_df['Description'] = pd.Categorical(plot_df['Description'], categories=sort_order, ordered=True)
plot_df = plot_df.sort_values('Description')

# Create 3-Column Subplots
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=("Total Earnings (£)", "Total Units", "Avg Unit Price (£)"),
    shared_yaxes=True, 
    horizontal_spacing=0.04
)

colors = {'Normal': '#636EFA', 'High-Value Outlier': '#EF553B'}

for segment in ['Normal', 'High-Value Outlier']:
    seg_data = plot_df[plot_df['Segment'] == segment]
    
    # Left Plot: Total Earnings (Sorted)
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Total_Earnings'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=True
    ), row=1, col=1)
    
    # Middle Plot: Total Units
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Total_Units'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=False
    ), row=1, col=2)
    
    # Right Plot: Average Price
    fig.add_trace(go.Bar(
        y=seg_data['Description'], x=seg_data['Avg_Price'], 
        name=segment, orientation='h', marker_color=colors[segment],
        legendgroup=segment, showlegend=False
    ), row=1, col=3)

fig.update_layout(
    title_text="Top 10 Stationery Products by Revenue: Earnings vs. Volume vs. Price",
    barmode='group', 
    template='plotly_white', 
    height=800,
    legend=dict(title="Segment", x=1.02, y=1)
)

fig.show()

In [25]:
# Aggregate by Segment and Category
cat_stats = df_segment_clean.groupby(['Category', 'Segment']).agg(
    Avg_UnitPrice=('UnitPrice', 'mean'),
    Avg_Qty=('Quantity', 'mean'),
    Total_Rev=('TotalCost', 'sum')
).reset_index()

# Filter for categories that both segments actually buy to ensure a fair comparison
common_cats = cat_stats.groupby('Category').filter(lambda x: x['Segment'].nunique() > 1)

# Get the top 10 categories by total revenue to keep the visualization clean
top_10_rev_cats = common_cats.groupby('Category')['Total_Rev'].sum().nlargest(10).index
plot_df = common_cats[common_cats['Category'].isin(top_10_rev_cats)]

# Visualize Avg Unit Price per Category by Segment
fig = px.bar(plot_df, 
             x='Category', 
             y='Avg_UnitPrice', 
             color='Segment',
             barmode='group',
             title='Average Unit Price per Category: Normal vs. Outlier',
             labels={'Avg_UnitPrice': 'Avg Price (£)'},
             template='plotly_white')

fig.show()


<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    
After removing all non product effect, the price between high value outlier and normal are about the same. 

</div>


<a id="sales-forecasting"></a>
# <p style="background-color: #81f1f3; font-family:calibri; color:dark; font-size:80%; font-family:Verdana; text-align:left;  padding: 20px;  border-radius:15px 15px;"> Sales Forecasting </p>

⬆️ [Tabel of Contents](#contents-label)

</div>


1. __Logistic Growth Model with Seasonal Adjustments__: . combines S-curve growth modeling with seasonal indexing to predict future trends that have a natural ceiling This is a great approach for forecasting demand or logistics data that has an upper limit (capacity) and periodic fluctuations.

2. __Holt-Winters Model__: The Holt-Winters model, or triple exponential smoothing, is a popular time-series forecasting technique that models data with both trends and seasonal patterns. It improves upon previous methods by smoothing three components: level, trend, and seasonality, using parameters (alpha, beta, gamma ). It is widely applied to sales, finance, and demand forecasting.

3. __Facebook Prophet__: Facebook Prophet is an open-source library for Python and R, developed by Meta (Facebook), designed for fast, automated, and accurate time-series forecasting. It is highly effective for business data with strong seasonal effects (daily, weekly, yearly) and multiple seasons of historical data, making it ideal for forecasting metrics like sales, web traffic, and capacity planning 

In [26]:
# Logistic Growth Model with Seasonal Adjustments


def get_logistic_forecast(category_name, df):
    # Resample to monthly
    series = df[df['Category'] == category_name].groupby('InvoiceDate')['Earnings'].sum().resample('MS').sum().fillna(0)
    
    # Logistic Simulation Parameters
    L = series.max() * 1.1  # Capacity (Cap)
    k = 0.02               # Growth rate
    x0 = len(series) / 2    # Midpoint
    
    # Project 12 months ahead
    future_steps = 12
    x_future = np.arange(len(series), len(series) + future_steps)
    
    # Calculate Forecast with Seasonality Multiplier (based on historical average)
    historical_avg = series.mean()
    seasonality = (series.tail(12).values / historical_avg) if len(series) >= 12 else np.ones(12)
    
    # Logistic Growth Formula: f(x) = L / (1 + e^(-k(x-x0)))
    forecast_values = (L / (1 + np.exp(-k * (x_future - x0)))) * seasonality
    
    # Ensure Floor of 0
    forecast_values = np.maximum(forecast_values, 0)
    
    forecast_index = pd.date_range(start=series.index[-1] + pd.DateOffset(months=1), periods=future_steps, freq='MS')
    
    # Save into DataFrame
    return pd.DataFrame({'Date': forecast_index, 'Projected_Revenue': forecast_values, 'Category': category_name}), series

# un Forecasts
hd_forecast, hd_actual = get_logistic_forecast('Home Decor', df_segment_clean)
kw_forecast, kw_actual = get_logistic_forecast('Kitchenware', df_segment_clean)

# Save to variable
final_forecast_variable = pd.concat([hd_forecast, kw_forecast])

# Create Plotly Visualization
fig = go.Figure()

# Home Decor
fig.add_trace(go.Scatter(x=hd_actual.index, y=hd_actual.values, name='Home Decor (Actual)', line=dict(color='#1f77b4', width=1)))
fig.add_trace(go.Scatter(x=hd_forecast['Date'], y=hd_forecast['Projected_Revenue'], name='Home Decor (Forecast)', line=dict(color='#1f77b4', width=3, dash='dot')))

# Kitchenware
fig.add_trace(go.Scatter(x=kw_actual.index, y=kw_actual.values, name='Kitchenware (Actual)', line=dict(color='#2ca02c', width=1)))
fig.add_trace(go.Scatter(x=kw_forecast['Date'], y=kw_forecast['Projected_Revenue'], name='Kitchenware (Forecast)', line=dict(color='#2ca02c', width=3, dash='dot')))

fig.update_layout(
    title='12-Month Logistic Revenue Projection (Clean Data)',
    xaxis_title='Timeline', yaxis_title='Monthly Earnings (£)',
    template='plotly_white', hovermode='x unified', height=600
)

fig.show()


<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    
1. By using the Logistic Growth approach, we’ve successfully stabilized the trend lines, ensuring both categories remain within realistic business bounds while capturing that massive Q4 surge

2. The £1M November Milestone: The combined forecast predicts your first £1,000,000+ month in November. This is a critical psychological and operational benchmark.

3. The December Drop: Note the sharp decline in December (£261k for Home Decor). This reflects the shipping cut-off dates for retail orders while consumers are still buying, your Outliers (wholesalers) have already finished their stocking cycles.

</div>

In [27]:

# 2. Prepare Series for Kitchenware and Home Decor
def get_monthly_series(category_name):
    cat_df = df_segment_clean[(df_segment_clean['Category'] == category_name)]
    return cat_df.groupby('InvoiceDate')['Earnings'].sum().resample('MS').sum().fillna(0)

kitchen_ts = get_monthly_series('Kitchenware')
home_decor_ts = get_monthly_series('Home Decor')

# 3. Forecast Logic
fig = go.Figure()
colors = {'Kitchenware': '#2ca02c', 'Home Decor': '#1f77b4'}

for name, ts in zip(['Kitchenware', 'Home Decor'], [kitchen_ts, home_decor_ts]):
    # Fit Model
    model = ExponentialSmoothing(ts, trend='multiplicative', seasonal='multiplicative', seasonal_periods=12).fit()
    forecast = model.forecast(12)
    forecast_index = pd.date_range(start=ts.index[-1] + pd.DateOffset(months=1), periods=12, freq='MS')
    
    # Actual Trace
    fig.add_trace(go.Scatter(x=ts.index, y=ts.values, name=f'{name} (Actual)', line=dict(color=colors[name])))
    # Forecast Trace
    fig.add_trace(go.Scatter(x=forecast_index, y=forecast.values, name=f'{name} (Forecast)', 
                             line=dict(color=colors[name], dash='dot')))



fig.update_layout(title='12-Month Projected Growth: Kitchenware vs. Home Decor',
                  xaxis_title='Date', yaxis_title='Revenue (£)',
                  template='plotly_white', hovermode='x unified')
fig.show()

In [28]:
df_segment_clean = df_segment_clean.reset_index(drop=True)

In [29]:
df_segment_clean = df_segment_clean.loc[:, ~df_segment_clean.columns.duplicated()]

In [30]:
df_segment_clean.groupby('InvoiceDate')['Earnings'].sum().reset_index()

,InvoiceDate,Earnings
0,2009-12-01 07:45:00,505.30
1,2009-12-01 07:46:00,145.80
2,2009-12-01 09:06:00,630.33
3,2009-12-01 09:08:00,310.75
4,2009-12-01 09:24:00,2286.24
...,...,...
40221,2011-12-09 12:23:00,124.60
40222,2011-12-09 12:25:00,140.64
40223,2011-12-09 12:31:00,329.05
40224,2011-12-09 12:49:00,339.20



<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    
This forecast suggests Kitchenware peaks in January (£46k) and declines through the year.

</div>

In [31]:
from prophet import Prophet

def get_prophet_forecast(category_name, df):
    # Filter and format for Prophet (requires 'ds' and 'y')
    cat_df = df[df['Category'] == category_name]
    ts = cat_df.groupby('InvoiceDate')['Earnings'].sum().reset_index()
    ts.columns = ['ds', 'y']
    
    # Set Logistic parameters: Cap at 20% above historical max, Floor at 0
    cap_limit = ts['y'].max() * 1.2
    ts['cap'] = cap_limit
    ts['floor'] = 0
    
    # Initialize and Fit
    model = Prophet(growth='linear', yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    model.fit(ts)
    
    # Create 12-month future dataframe
    future = model.make_future_dataframe(periods=12, freq='MS')
    future['cap'] = cap_limit
    future['floor'] = 0
    
    forecast = model.predict(future)
    forecast['Category'] = category_name
    return forecast, ts

# Run for both categories
hd_forecast, hd_actual = get_prophet_forecast('Home Decor', df_segment_clean)
kw_forecast, kw_actual = get_prophet_forecast('Kitchenware', df_segment_clean)

# Save combined forecast data (Next 12 Months) to variable
final_prophet_forecast = pd.concat([
    hd_forecast[['ds', 'yhat', 'Category']].tail(12),
    kw_forecast[['ds', 'yhat', 'Category']].tail(12)
])

15:10:43 - cmdstanpy - INFO - Chain [1] start processing
15:10:49 - cmdstanpy - INFO - Chain [1] done processing
15:10:59 - cmdstanpy - INFO - Chain [1] start processing
15:11:02 - cmdstanpy - INFO - Chain [1] done processing


In [32]:
fig = go.Figure()

# --- Home Decor ---
# Forecast
fig.add_trace(go.Scatter(x=hd_forecast['ds'], y=hd_forecast['yhat'], name='Home Decor (Prophet)', 
                         line=dict(color='#1f77b4', width=3)))

# --- Kitchenware ---
# Actuals
#fig.add_trace(go.Scatter(x=kw_actual['ds'], y=kw_actual['y'], name='Kitchenware (Actual)', 
#                         mode='markers', marker=dict(color='#2ca02c', size=4)))
# Forecast
fig.add_trace(go.Scatter(x=kw_forecast['ds'], y=kw_forecast['yhat'], name='Kitchenware (Prophet)', 
                         line=dict(color='#2ca02c', width=3)))

fig.update_layout(
    title='<b>Combined Revenue Forecast: Logistic Prophet Model</b>',
    xaxis_title='Date',
    yaxis_title='Revenue (£)',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()


<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>
    
Even though November has the highest total revenue, Facebook prophet predicts the October has the highest revenue. This suggests that in October, customers (likely High-Value Outliers) are placing their most comprehensive, expensive orders to prepare for the holiday rush.

</div>

In [33]:
## Seasonal Purchase: Normal vs High-Value

# 1. Prepare the data: Aggregate by Month and Segment
df_segment_clean['MonthYear'] = df_segment_clean['InvoiceDate'].dt.to_period('M').astype(str)
seasonal_data = df_segment_clean.groupby(['MonthYear', 'Segment'])['TotalCost'].sum().reset_index()

# 2. Pivot for easier plotting
pivot_seasonal = seasonal_data.pivot(index='MonthYear', columns='Segment', values='TotalCost').fillna(0)

# 3. Build the Line Chart
fig = go.Figure()

# Normal Customer Line
fig.add_trace(go.Scatter(
    x=pivot_seasonal.index, 
    y=pivot_seasonal['Normal'],
    name='Normal (Retail)',
    line=dict(color='#636EFA', width=3),
    mode='lines+markers'
))

# High-Value Outlier Line
fig.add_trace(go.Scatter(
    x=pivot_seasonal.index, 
    y=pivot_seasonal['High-Value Outlier'],
    name='High-Value (Wholesale)',
    line=dict(color='#EF553B', width=3, dash='dash'),
    mode='lines+markers'
))

# 4. Layout Updates
fig.update_layout(
    title='Seasonal Purchasing Pattern: Retail vs. Wholesale Customers',
    xaxis_title='Month',
    yaxis_title='Total Revenue (£)',
    hovermode='x unified',
    template='plotly_white',
    height=600
)

fig.show()

In [34]:
# --- Prepare Historical Data ---
# Ensure InvoiceDate is datetime and create Monthly series per segment
df_segment_clean['InvoiceDate'] = pd.to_datetime(df_segment_clean['InvoiceDate'])
segments = ['High-Value Outlier', 'Normal']
all_data_list = []

for seg in segments:
    # Historical monthly aggregation
    hist_series = df_segment_clean[df_segment_clean['Segment'] == seg].groupby(
        pd.Grouper(key='InvoiceDate', freq='MS'))['TotalCost'].sum().fillna(0)
    
    # Fit Holt-Winters (Triple Exponential Smoothing)
    # Using 'add' trend and seasonality for stable retail patterns
    model = ExponentialSmoothing(hist_series, trend='add', seasonal='add', seasonal_periods=12).fit()
    
    # Forecast 12 Months (2012)
    forecast_values = model.forecast(12)
    forecast_dates = pd.date_range(start=hist_series.index[-1] + pd.DateOffset(months=1), periods=12, freq='MS')
    
    # Combine into a single segment dataframe
    hist_df = hist_series.reset_index()
    hist_df.columns = ['Date', 'Value']
    hist_df['Type'] = 'Actual'
    
    fore_df = pd.DataFrame({'Date': forecast_dates, 'Value': forecast_values, 'Type': 'Forecast'})
    
    seg_df = pd.concat([hist_df, fore_df])
    seg_df['Segment'] = seg
    all_data_list.append(seg_df)

# Final Combined DataFrame
full_df = pd.concat(all_data_list)
full_df['Month'] = full_df['Date'].dt.strftime('%b') # Month Name
full_df['Year'] = full_df['Date'].dt.year

# Sort months chronologically for the plot
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

In [35]:
# Create Subplots: One for High-Value, One for Normal
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, 
                    subplot_titles=("High-Value Outlier: YoY Monthly Revenue", "Normal: YoY Monthly Revenue"),
                    vertical_spacing=0.15)

colors = {2010: '#636EFA', 2011: '#EF553B', 2012: '#00CC96'}

for i, seg in enumerate(segments):
    seg_data = full_df[full_df['Segment'] == seg]
    
    for year in [2010, 2011, 2012]:
        year_data = seg_data[seg_data['Year'] == year]
        
        # Ensure months are in order
        year_data = year_data.set_index('Month').reindex(month_order).reset_index()
        
        fig.add_trace(
            go.Bar(
                x=year_data['Month'],
                y=year_data['Value'],
                name=f"{year} ({'Forecast' if year == 2012 else 'Actual'})",
                marker_color=colors[year],
                legendgroup=str(year),
                showlegend=(i == 0) # Only show legend once
            ),
            row=i+1, col=1
        )

fig.update_layout(
    height=900,
    title_text="Segmented Year-Over-Year Monthly Comparison (Holt-Winters 2012)",
    barmode='group',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [36]:
import plotly.graph_objects as go
import pandas as pd

# (Assuming your provided data is loaded into a DataFrame named 'df_result')
# Mapping colors for the years
colors = {2010: '#636EFA', 2011: '#EF553B', 2012: '#00CC96'}
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig = go.Figure()

for segment in ['High-Value Outlier', 'Normal']:
    for year in [2010, 2011, 2012]:
        mask = (full_df['Segment'] == segment) & (full_df['Year'] == year)
        data = full_df[mask].set_index('Month').reindex(month_order).reset_index()
        
        fig.add_trace(go.Bar(
            x=data['Month'],
            y=data['Value'],
            name=f"{year} {segment}",
            visible=True if segment == 'High-Value Outlier' else 'legendonly',
            marker_color=colors[year],
            legendgroup=str(year)
        ))

fig.update_layout(
    title='<b>Year-Over-Year Monthly Revenue: Actuals vs Holt-Winters Forecast</b>',
    xaxis_title='Month',
    yaxis_title='Revenue (£)',
    barmode='group',
    template='plotly_white',
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.7, y=1.2,
        buttons=[
            dict(label="High-Value Outliers", method="update", args=[{"visible": [True, True, True, False, False, False]}]),
            dict(label="Normal Customers", method="update", args=[{"visible": [False, False, False, True, True, True]}]),
            dict(label="Compare All", method="update", args=[{"visible": [True]*6}])
        ]
    )]
)

fig.show()

<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h5><strong>Comments</strong></h5>

__Triple Exponential Smoothing (Holt-Winters) pattern__. The forecast for 2012 has effectively "learned" the seasonality of the business, particularly the massive peak in November and the standard "slump" in February and April.

__The November Peak__: The model projects £743k for Nov 2012. This is higher than both 2010 (£731k) and 2011 (£685k).

__The "slump" protection__: Both segments show a massive drop-off in April (£299k and £167k). This is your optimal time for warehouse maintenance, inventory audits, or staff vacations before the mid-year climb begins.

__High-Value Outliers__ This segment is remarkably stable. The forecast predicts that 2012 will closely mirror the 2010/2011 cycle but with a slight upward trend in the "off-peak" months.

__The November Peak__: The model projects £743k for Nov 2012. This is higher than both 2010 (£731k) and 2011 (£685k).

__Inventory Indicator__: Because this segment is so consistent, we can safely set "Safety Stock" levels for high-volume items (like the Regency Cakestand) based on a 5-10% growth buffer over 2011 actuals.

__Normal Segment__ (The Retail Surge) The "Normal" segment shows more dramatic seasonal swings. Look at the jump in September: in 2010 it was £273k, in 2011 it jumped to £354k, and the model projects a stabilization at £272k for 2012 (suggesting 2011 might have had an outlier event in September).

__The Q4 retail rush__: The gap between October (£418k) and November (£438k) is much narrower for Normal customers than for Outliers.

__Operational Insight__: While Outliers start dropping off in December, the Normal segment remains relatively active (£164k forecast), likely due to last-minute holiday shopping.

</div>

<a id="customer-products-recommendation"></a>
# <p style="background-color: #81f1f3; font-family:calibri; color:dark; font-size:80%; font-family:Verdana; text-align:left;  padding: 20px;  border-radius:15px 15px;"> Customer Products Recommendation </p>

⬆️ [Tabel of Contents](#contents-label)
</div>

By using the __Apriori Algorithm__ or __Association Rules__, we can find products that are frequently bought together and identify "gaps" in your top customers' purchasing history.

Apriori Algorithm is a basic method used in data analysis to find groups of items that often appear together in large sets of data. It helps to discover useful patterns or rules about how items are related which is particularly valuable in market basket analysis.

A high Lift score (e.g., > 5.0) tells us that when a customer buys Product A, they are significantly more likely to buy Product B than the average shopper.  We will focus on "Lift"—the gold standard for identifying hidden relationships.

In [37]:
df_segment_clean[df_segment_clean['Segment']=='High-Value Outlier']['Customer ID'].unique()

<IntegerArray>
[13078, 18102, 12682, 18087, 14110, 13758, 15413, 17865, 13767, 17238,
 ...
 12590, 12757, 16446, 15098, 12830, 14096, 17509, 12536, 18139, 16000]
Length: 693, dtype: Int64

In [ ]:
from scipy.sparse import csr_matrix

def get_fast_recommendations(df, target_ids, min_lift=3.0, min_conf=0.4):
    # 1. Filter for the segment and simplify columns
    hv_df = df[df['Segment'] == 'High-Value Outlier'][['Invoice', 'Description', 'Customer ID']]
    
    # 2. Vectorize the Descriptions (Map names to IDs for matrix math)
    hv_df['Description'] = hv_df['Description'].astype('category')
    item_codes = hv_df['Description'].cat.codes
    item_names = hv_df['Description'].cat.categories
    num_items = len(item_names)
    
    # 3. Create a Sparse Basket Matrix (Invoice x Item)
    # This is memory-efficient and much faster for math
    invoice_codes = hv_df['Invoice'].astype('category').cat.codes
    rows = invoice_codes
    cols = item_codes
    data = np.ones(len(hv_df))
    
    # Use 'clip' to ensure it's binary (1 or 0)
    sparse_basket = csr_matrix((data, (rows, cols)), shape=(len(invoice_codes.unique()), num_items))
    sparse_basket.data = np.ones_like(sparse_basket.data) 

    #  Calculate Co-occurrence and Support using Sparse Math
    item_support_counts = np.array(sparse_basket.sum(axis=0)).flatten()
    total_invoices = sparse_basket.shape[0]
    
    # Only calculate affinities for your target customers to save time
    recommendations = []
    
    for cust_id in target_ids:
        # Get items this customer HAS purchased
        cust_items_codes = hv_df[hv_df['Customer ID'] == cust_id]['Description'].cat.codes.unique()
        cust_items_codes = [c for c in cust_items_codes if c != -1]
        
        if not cust_items_codes:
            continue

        # For each item the customer has, find the 'friends' of that item
        for code_a in cust_items_codes:
            # Get the row of the co-occurrence matrix for item_a only
            # (sparse_basket.T[code_a] * sparse_basket)
            col_a = sparse_basket[:, code_a]
            co_occurrences_with_a = col_a.T.dot(sparse_basket).toarray().flatten()
            
            # Probability of B given A (Confidence)
            confidences = co_occurrences_with_a / item_support_counts[code_a]
            
            # Find indices where confidence is high
            high_conf_indices = np.where(confidences >= min_conf)[0]
            
            for code_b in high_conf_indices:
                if code_b not in cust_items_codes:
                    # Calculate Lift
                    support_b = item_support_counts[code_b] / total_invoices
                    lift = confidences[code_b] / support_b
                    
                    if lift >= min_lift:
                        recommendations.append({
                            'Customer ID': cust_id,
                            'Trigger': item_names[code_a],
                            'Missed_Item': item_names[code_b],
                            'Confidence': round(confidences[code_b], 2),
                            'Lift': round(lift, 2)
                        })
    
    return pd.DataFrame(recommendations).drop_duplicates(['Customer ID', 'Missed_Item'])

# Run it
target_list = [13078, 18102, 12682, 18087, 14110, 13758, 15413, 17865, 13767, 17238]
#target_list = [18102]
fast_results = get_fast_recommendations(df_segment_clean, target_list)
#print(fast_results.sort_values('Lift', ascending=False).head(10))

     Customer ID                        Trigger  \
120        18102        ZINC POLICE BOX LANTERN   
959        17238  WHITE HEART OF GLASS BRACELET   
960        17238  WHITE HEART OF GLASS BRACELET   
926        17238    PINK CRYSTAL+GLASS BRACELET   
962        17238  WHITE HEART OF GLASS BRACELET   
928        17238    PINK CRYSTAL+GLASS BRACELET   
898        17238        BLACK GEMSTONE BRACELET   
697        15413        BEECH WOOD PHOTO FRAME    
698        15413        BEECH WOOD PHOTO FRAME    
927        17238    PINK CRYSTAL+GLASS BRACELET   

                             Missed_Item  Confidence     Lift  
120  F FAIRY POTPOURRI CUSHIONS LAVENDER        0.50  4413.00  
959    LAZER CUT NECKLACE W PASTEL BEADS        0.67  2942.00  
960   LONG SILVER NECKLACE PASTEL FLOWER        0.67  2942.00  
926        GREEN HEART OF GLASS BRACELET        0.67  2942.00  
962         PINK HEART OF GLASS BRACELET        0.67  2353.60  
928    TURQUOISE HEART OF GLASS BRACELET        0.67  

In [39]:
# Create a polished, styled table for the recommendations
styled_recommendations = fast_results.sort_values(['Customer ID', 'Lift'], ascending=[True, False]).copy()

# Rename columns f
styled_recommendations.columns = ['Customer ID', 'Based on Purchase of', 'Recommended Miss', 'Probability (Conf)', 'Strength (Lift)']

# Apply styling
styled_output = styled_recommendations.style.background_gradient(cmap='Blues', subset=['Probability (Conf)'])\
    .background_gradient(cmap='YlOrRd', subset=['Strength (Lift)'])\
    .format({'Probability (Conf)': '{:.0%}', 'Strength (Lift)': '{:.2f}'})\
    .set_caption("High-Value 'Missed Opportunity' Matrix")

styled_output

,Customer ID,Based on Purchase of,Recommended Miss,Probability (Conf),Strength (Lift)
468,12682,DOILY THANK YOU CARD,PAISLEY PARK CARD,42%,675.67
479,12682,PAPER LANTERN 9 POINT SNOW STAR,PAPER LANTERN 9 POINT SNOW STAR,42%,668.64
451,12682,EMBROIDERED RIBBON REEL REBECCA,EMBROIDERED RIBBON REEL RACHEL,41%,605.71
336,12682,PIN CUSHION RUSSIAN DOLL RED,PIN CUSHION RUSSIAN DOLL BLUE,43%,548.20
457,12682,OVAL MINI PORTRAIT FRAME,SQUARE MINI PORTRAIT FRAME,52%,513.68
455,12682,EMBROIDERED RIBBON REEL REBECCA,EMBROIDERED RIBBON REEL SUSIE,53%,467.26
454,12682,EMBROIDERED RIBBON REEL REBECCA,EMBROIDERED RIBBON REEL SOPHIE,41%,454.28
456,12682,OVAL MINI PORTRAIT FRAME,HEART MINI PORTRAIT FRAME,76%,448.30
452,12682,EMBROIDERED RIBBON REEL REBECCA,EMBROIDERED RIBBON REEL ROSIE,47%,437.20
453,12682,EMBROIDERED RIBBON REEL REBECCA,EMBROIDERED RIBBON REEL SALLY,59%,432.65


In [44]:
import datetime as dt

# 1. Reference date (assuming the day after the last transaction in the dataset)
snapshot_date = df_segment_clean['InvoiceDate'].max() + dt.timedelta(days=1)

# 2. Calculate RFM Metrics per High-Value Customer
churn_df = df_segment_clean[df_segment_clean['Segment'] == 'High-Value Outlier'].groupby('Customer ID').agg({
    'InvoiceDate': [lambda x: (snapshot_date - x.max()).days, # Recency
                    lambda x: (x.max() - x.min()).days],     # Tenure
    'Invoice': 'count',                                    # Frequency
    'TotalCost': 'sum'                                       # Monetary
}).reset_index()

churn_df.columns = ['Customer ID', 'Recency', 'Tenure', 'Frequency', 'Monetary']

# 3. Calculate Average Purchase Interval (API)
# How many days usually pass between their orders?
churn_df['Avg_Order_Interval'] = churn_df['Tenure'] / churn_df['Frequency']

# 4. Churn Risk Factor
# Risk = Current Recency / Average Interval
# A factor > 2.0 means they have missed 2+ expected purchase cycles.
churn_df['Risk_Factor'] = churn_df['Recency'] / churn_df['Avg_Order_Interval']
churn_df['Risk_Status'] = pd.cut(churn_df['Risk_Factor'], 
                                 bins=[0, 1, 2, 5, float('inf')], 
                                 labels=['Active', 'Warning', 'High Risk', 'Lost'])

# Sort by Risk to see who to call first
churn_risk_report = churn_df.sort_values('Risk_Factor', ascending=False)

In [42]:
churn_risk_report

,Customer ID,Recency,Tenure,Frequency,Monetary,Avg_Order_Interval,Risk_Factor,Risk_Status
493,16446,206,0,2,2.90,0.000000,inf,Lost
44,12590,211,0,67,9341.26,0.000000,inf,Lost
450,16000,3,0,9,12393.70,0.000000,inf,Lost
676,18139,18,0,159,8438.34,0.000000,inf,Lost
356,15098,182,0,3,39916.50,0.000000,inf,Lost
...,...,...,...,...,...,...,...,...
547,16954,1,689,206,6081.88,3.344660,0.298984,Active
216,13953,7,721,25,6640.68,28.840000,0.242718,Active
655,17949,1,736,153,107833.48,4.810458,0.207880,Active
416,15694,1,731,142,9963.48,5.147887,0.194254,Active


In [43]:
# 1. Merge the dataframes
dashboard = pd.merge(
    churn_risk_report[['Customer ID', 'Risk_Status', 'Risk_Factor']], 
    fast_results[['Customer ID', 'Missed_Item', 'Confidence']], 
    on='Customer ID', 
    how='left'
)

# 2. Keep only the highest confidence recommendation per customer
dashboard = dashboard.sort_values(['Customer ID', 'Confidence'], ascending=[True, False])
dashboard = dashboard.drop_duplicates('Customer ID')

# 3. Filter for your target list to focus the report
target_dashboard = dashboard[dashboard['Customer ID'].isin(target_list)]

# 4. Final Formatting for Export
def color_risk(val):
    color = 'green' if val == 'Active' else 'orange' if val == 'Warning' else 'red' if val == 'High Risk' else 'gray'
    return f'color: {color}; font-weight: bold'

final_view = target_dashboard.style.map(color_risk, subset=['Risk_Status'])\
    .format({'Risk_Factor': '{:.2f}', 'Confidence': '{:.0%}'})\
    .set_caption("Consolidated Account Health & Cross-Sell Dashboard")

final_view

,Customer ID,Risk_Status,Risk_Factor,Missed_Item,Confidence
676,12682,Lost,5.12,REGENCY TEA PLATE ROSES,90%
1061,13078,High Risk,3.23,REGENCY TEA PLATE PINK,75%
760,13758,High Risk,4.69,GIN & TONIC DIET GREETING CARD,73%
1180,13767,High Risk,2.08,ANTIQUE GLASS DRESSING TABLE POT,60%
1148,14110,High Risk,2.18,STRAWBERRY LUNCH BOX WITH CUTLERY,69%
26,15413,Lost,368.09,GLASS CHALICE BLUE SMALL,67%
856,17238,High Risk,4.64,PURPLE GEMSTONE BRACELET,80%
233,17865,Lost,44.18,SPACEBOY CHILDRENS BOWL,83%
109,18087,Lost,56.51,FELTCRAFT DOLL ROSIE,61%
1348,18102,Warning,1.41,LUNCH BAG SUKI DESIGN,75%


<div style="border-radius:10px; padding: 20px; box-sizing: border-box; background-color: #c9fdb9; font-size:100%; border: 1px solid #ccc;">
     <h2><strong>Conclusions</strong></h2>

1. __Identifying the Business Core__:
The analysis successfully moved beyond a "one-size-fits-all" approach by distinguishing between Normal retail customers and High-Value Outliers. Recognizing that a small fraction of the customer base—likely wholesalers and professional contractors—drives a disproportionate share of revenue allows for more efficient resource allocation. This segmentation ensures that high-touch service and specialized logistics are reserved for the accounts that underpin the company's financial stability.

2. __Strategic Planning__:
Utilizing both Prophet (for growth modeling) and Holt-Winters (for seasonal baseline accuracy), the report provides a 12-month roadmap for 2012.

     - __Home Decor__ remains the primary revenue engine with a massive projected surge in Q4.

     - __Kitchenware__ shows a front-loaded seasonal pattern, peaking in January.

     These forecasts provide the necessary lead times for procurement and warehouse staffing, ensuring the infrastructure is prepared for the predicted __£1M+__ revenue month in November.

3.  __Closing the Gap__:
The Market Basket Analysis, optimized via sparse matrix calculations, identified high-affinity product clusters. By cross-referencing these statistical "ideal baskets" with individual customer histories, we identified specific "Missed Opportunities." This system turns every re-engagement effort into a personalized pitch, utilizing "High Lift" items to increase the Average Transaction Value (ATV) and provide a meaningful "hook" for win-back campaigns.

4. __Quantifying Loyalty and Risk__:
The Recency, Frequency, and Monetary (RFM) framework provided a standardized health check for every account. By calculating the Average Purchase Interval (API), we moved from static metrics to dynamic "Risk Factors." This allows the business to identify "Warning" and "High Risk" customers based on their individual habits rather than arbitrary dates. This section transformed raw transaction history into a prioritized "Account Health Dashboard" for the sales team.

<br>

__Executive Summary Statement__:

The integration of these four modules creates a closed-loop system: Forecasting tells us when the demand will hit, Segmentation tells us who will drive it, RFM tells us who is at risk of leaving, the Recommendation System tells us exactly what to sell to keep them engaged. By executing on these insights, the business is positioned to maximize its 2012 growth while systematically reducing customer churn.


</div>